## Homework 2: Canny edge detector

**Release date:** Feb 27st 2026

**Due date:** Mar 12th 2026

The goal of the assignment is to implement a Canny edge detector. You should return the completed notebook, including answers and illustrations.

If you are using [anaconda](https://www.anaconda.com/distribution/) you will have necessary libraries, if not, you may need to install them.

**Load and visualize image**

In [1]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import scipy.ndimage as ndimage #image processing library

# by default, the grayscale images are displayed with the jet colormap: use grayscale instead
plt.rcParams['image.cmap'] = 'gray'

def rgb2gray(rgb):
    r, g, b = rgb[:,:,0], rgb[:,:,1], rgb[:,:,2]
    gray = 0.2989 * r + 0.5870 * g + 0.1140 * b
    return gray

name = 'example_image.jpg.jpeg'
img = plt.imread(name)
img = rgb2gray(img)
plt.figure(figsize=(10,10)) # this allows you to control the size of the displayed image
plt.imshow(img)


FileNotFoundError: [Errno 2] No such file or directory: 'example_image.jpg'

**Detailed instructions:**

a- Compute a binary image corresponding to thresholding the norm of the gradient. You may use the function `ndimage.gaussian_filter` to compute the derivative of gaussian filter (see [docs](https://docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.gaussian_filter.html)). Discuss the parameters (there are two) and their influence on the results.

**Discussion of the two key parameters:**
- `sigma`: standard deviation of the Gaussian smoothing. Larger `sigma` increases smoothing before differentiation, which suppresses noise but also removes fine details and can shift/thicken edges. Smaller `sigma` preserves details but is more noise-sensitive.
- `threshold` (applied on gradient norm): controls edge selectivity. A higher threshold keeps only strong edges (fewer false positives, but misses weak contours); a lower threshold keeps more contours (better recall, but more noise).


In [2]:
# Gradient computation + simple thresholding (step a)

def compute_gradient(image, sigma=1.0):
    """Return horizontal/vertical derivatives, gradient norm and direction."""
    gx = ndimage.gaussian_filter(image, sigma=sigma, order=[0, 1], mode='reflect')
    gy = ndimage.gaussian_filter(image, sigma=sigma, order=[1, 0], mode='reflect')
    g_norm = np.hypot(gx, gy)
    g_theta = np.arctan2(gy, gx)
    return gx, gy, g_norm, g_theta


def threshold_gradient(g_norm, threshold):
    """Binary edge map by thresholding gradient magnitude."""
    return (g_norm >= threshold).astype(np.uint8)

# quick visualization for step (a)
_, _, g_norm_demo, _ = compute_gradient(img, sigma=1.4)
thr_demo = np.percentile(g_norm_demo, 80)
edge_demo = threshold_gradient(g_norm_demo, thr_demo)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(g_norm_demo)
ax[0].set_title('Gradient norm')
ax[0].axis('off')
ax[1].imshow(edge_demo)
ax[1].set_title(f'Thresholded norm (t={thr_demo:.2f})')
ax[1].axis('off')
plt.show()


b- Write a function `nms(g_norm,g_theta)` which takes as input the gradient norm and direction and outputs a binary image with value 1 only for pixels that correspond to a maximum in the direction of the gradient.

In [3]:
# Non-maximum suppression (step b)
def nms(g_norm, g_theta):
    """Keep local maxima of gradient norm along gradient direction."""
    rows, cols = g_norm.shape
    out = np.zeros_like(g_norm, dtype=np.uint8)

    # map angle from radians to [0,180)
    angle = (np.rad2deg(g_theta) + 180) % 180

    for i in range(1, rows - 1):
        for j in range(1, cols - 1):
            a = angle[i, j]
            q = r = 0.0

            # 0 degrees
            if (0 <= a < 22.5) or (157.5 <= a < 180):
                q = g_norm[i, j + 1]
                r = g_norm[i, j - 1]
            # 45 degrees
            elif 22.5 <= a < 67.5:
                q = g_norm[i + 1, j - 1]
                r = g_norm[i - 1, j + 1]
            # 90 degrees
            elif 67.5 <= a < 112.5:
                q = g_norm[i + 1, j]
                r = g_norm[i - 1, j]
            # 135 degrees
            elif 112.5 <= a < 157.5:
                q = g_norm[i - 1, j - 1]
                r = g_norm[i + 1, j + 1]

            if g_norm[i, j] >= q and g_norm[i, j] >= r:
                out[i, j] = 1

    return out

nms_demo = nms(g_norm_demo, compute_gradient(img, sigma=1.4)[3])
plt.figure(figsize=(6,6))
plt.imshow(nms_demo)
plt.title('NMS maxima mask')
plt.axis('off')
plt.show()


c- Combine step 'a' and 'b' to extract edges with a gradient norm larger than a given threshold.

In [4]:
# Combine (a) + (b): thresholded edges after NMS (step c)
def edges_with_threshold(image, sigma=1.2, threshold=None, percentile=85):
    _, _, g_norm, g_theta = compute_gradient(image, sigma=sigma)
    maxima = nms(g_norm, g_theta)

    if threshold is None:
        threshold = np.percentile(g_norm, percentile)

    edges = ((g_norm >= threshold) & (maxima == 1)).astype(np.uint8)
    return edges, g_norm, g_theta, threshold

edges_c, g_norm_c, g_theta_c, t_c = edges_with_threshold(img, sigma=1.2, percentile=85)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(img)
ax[0].set_title('Input image')
ax[0].axis('off')
ax[1].imshow(edges_c)
ax[1].set_title(f'Step (c): edges, t={t_c:.2f}')
ax[1].axis('off')
plt.show()


d- Add the hysteresis thresholding to step 'c' to implement a function computing the Canny edges.

Here is one way to do the hysteresis thresholding. Apply step 'c' with two thresholds $t_1 < t_2$. This yields a set of "strong pixels" (large threshold) and "weak pixels" (small threshold). Initialize a list of edge pixels by including only the set of strong pixels. For each pixel in this list, check if its neighboors are weak pixels. If they are, add them to the list of pixels to visit.

In [5]:
# Hysteresis thresholding + full canny-style edges (step d)
def hysteresis_threshold(g_norm, maxima_mask, t_low, t_high):
    strong = (g_norm >= t_high) & (maxima_mask == 1)
    weak = (g_norm >= t_low) & (maxima_mask == 1)

    result = np.zeros_like(g_norm, dtype=np.uint8)
    visited = np.zeros_like(g_norm, dtype=bool)

    stack = [tuple(p) for p in np.argwhere(strong)]
    for p in stack:
        result[p] = 1
        visited[p] = True

    neighbors = [(-1,-1), (-1,0), (-1,1), (0,-1), (0,1), (1,-1), (1,0), (1,1)]

    while stack:
        i, j = stack.pop()
        for di, dj in neighbors:
            ni, nj = i + di, j + dj
            if 0 <= ni < g_norm.shape[0] and 0 <= nj < g_norm.shape[1]:
                if weak[ni, nj] and not visited[ni, nj]:
                    visited[ni, nj] = True
                    result[ni, nj] = 1
                    stack.append((ni, nj))

    return result


def canny_edges(image, sigma=1.2, t_low=None, t_high=None, low_pct=75, high_pct=90):
    _, _, g_norm, g_theta = compute_gradient(image, sigma=sigma)
    maxima = nms(g_norm, g_theta)

    if t_low is None:
        t_low = np.percentile(g_norm, low_pct)
    if t_high is None:
        t_high = np.percentile(g_norm, high_pct)

    if t_low >= t_high:
        raise ValueError('t_low must be smaller than t_high')

    edges = hysteresis_threshold(g_norm, maxima, t_low, t_high)
    return edges, g_norm, (t_low, t_high)

edges_d, g_norm_d, (tl, th) = canny_edges(img, sigma=1.4, low_pct=70, high_pct=90)

fig, ax = plt.subplots(1, 2, figsize=(12,5))
ax[0].imshow(g_norm_d)
ax[0].set_title('Gradient norm')
ax[0].axis('off')
ax[1].imshow(edges_d)
ax[1].set_title(f'Canny-like edges (t_low={tl:.2f}, t_high={th:.2f})')
ax[1].axis('off')
plt.show()


e- Run your code on at least four images of your own choosing. Use different parameters and comment on their effects.

In [ ]:
# Experiments on four additional images generated from the given image (step e)
rng = np.random.default_rng(0)

images = {
    'Original': img,
    'Rotated (+20°)': ndimage.rotate(img, 20, reshape=False, mode='reflect'),
    'Blurred (sigma=2)': ndimage.gaussian_filter(img, sigma=2.0),
    'Noisy (+Gaussian noise)': np.clip(img + 0.08 * rng.standard_normal(img.shape), 0, 1),
    'High-contrast': np.clip((img - img.mean()) * 1.6 + img.mean(), 0, 1),
}

params = {
    'Original': dict(sigma=1.4, low_pct=70, high_pct=90),
    'Rotated (+20°)': dict(sigma=1.4, low_pct=70, high_pct=90),
    'Blurred (sigma=2)': dict(sigma=1.0, low_pct=68, high_pct=88),
    'Noisy (+Gaussian noise)': dict(sigma=2.0, low_pct=75, high_pct=92),
    'High-contrast': dict(sigma=1.2, low_pct=72, high_pct=90),
}

n = len(images)
fig, axes = plt.subplots(n, 2, figsize=(12, 4*n))

for idx, (title, im) in enumerate(images.items()):
    p = params[title]
    e, _, (tl, th) = canny_edges(im, **p)

    axes[idx, 0].imshow(im)
    axes[idx, 0].set_title(title)
    axes[idx, 0].axis('off')

    axes[idx, 1].imshow(e)
    axes[idx, 1].set_title(f"Edges: sigma={p['sigma']}, t_low={tl:.2f}, t_high={th:.2f}")
    axes[idx, 1].axis('off')

plt.tight_layout()
plt.show()

print('Comments on parameter effects:')
print('- Larger sigma reduces noise (helpful for noisy image) but removes fine details.')
print('- Increasing thresholds keeps only strongest contours and reduces clutter.')
print('- Lower thresholds recover weak edges but can introduce texture/noise edges.')
print('- Rotating image mostly preserves edge quality (orientation-invariant gradient norm).')
